# SEBAL Notebook 6: Refined Fluxes and ET Products
**Scene:** Landsat 7 (2020-01-19)  
**Region:** Mississippi Delta (EPSG:32615)

This notebook applies the stability-corrected aerodynamic resistance and calibrated dT relation to compute refined SEBAL energy balance components for the Mississippi Delta study area. The workflow generates the corrected dT raster, refined sensible heat flux (H), latent heat flux (LE), evaporative fraction (EF), and evapotranspiration products.

## 1. Import Required Python Libraries and Utility Functions

In [1]:
import os
import rasterio
import numpy as np
from Utility import get_file_dataframe
from Utility import load_rasters
from Utility import calculate_NDVI
from Utility import calculate_albedo
from Utility import extract_mask_mean
from Utility import read_raster
from Utility import write_raster
from Utility import check_raster

## 2. Set SEBAL working directory

In [2]:
# move from 00_scripts → project root
os.chdir("..")

print("Working directory:", os.getcwd())

Working directory: G:\Collaborations\Mentee\UF_Anitha Madapakula\Scripts\Python\SEBAL_20200119_MSDelta


## 3.  Define core directories and inputs

In [3]:
# === Core directories and scene settings (used across notebooks) ===
clip_dir = "03_clip_landsat"
# output folder
out_dir = "04_indices"
met_dir     = "03_processed_met"
os.makedirs(out_dir, exist_ok=True)
region = 'MSDelta'
proc_dir = "03_processed_met"
date = '20200119'
hour_str = '16'

## 4. Define paths for required rasters

In [4]:
rn_path   = f"{out_dir}/Rn_{date}_{hour_str}Z_{region}.tif"
g_path    = f"{out_dir}/G_{date}_{hour_str}Z_{region}.tif"
lst_path  = f"{out_dir}/LST_C_{date}_{region}.tif"
ndvi_path = f"{out_dir}/NDVI_{date}_{region}.tif"
alb_path  = f"{out_dir}/ALBEDO_{date}_{region}.tif"
TA_path   = os.path.join(met_dir, f"TA_{date}_{hour_str}Z_{region}.tif")
U10_path  = os.path.join(met_dir, f"U10_{date}_{hour_str}Z_{region}.tif")
V10_path  = os.path.join(met_dir, f"V10_{date}_{hour_str}Z_{region}.tif")
wind_path  = os.path.join(met_dir, f"Wind10_{date}_{hour_str}Z_{region}.tif")

## 5. Load input raster

In [5]:
# load net radiation raster
Rn, profile, rn_nodata = read_raster(rn_path)

# load soil heat flux raster
G, _, g_nodata = read_raster(g_path)

# load land surface temperature raster
LstC, _, lst_nodata = read_raster(lst_path)

# load NDVI raster
Ndvi, _, ndvi_nodata = read_raster(ndvi_path)

# load albedo raster
Alb, _, alb_nodata = read_raster(alb_path)

# load meteorology
TA, _, _  = read_raster(TA_path)
U10, _, _ = read_raster(U10_path)
V10, _, _ = read_raster(V10_path)
U10_mag, _, _ = read_raster(wind_path)

# create physically valid NDVI copy for anchor-pixel selection
Ndvi2 = np.where((Ndvi > -1.0) & (Ndvi < 1.0), Ndvi, np.nan)

# build valid mask
valid = (Rn > -9999) & (G > -9999) & (LstC > -9999) & (Ndvi > -9999)

print("Valid pixels:", int(valid.sum()))

Valid pixels: 15410408


## 6. Compute friction velocity ($u_*$)
$$
u_* = \frac{kU_10}{ln(\frac{10}{z_0})}
$$

In [6]:
k = 0.41
z0 = 0.1   # initial roughness length (m)

ustar = np.full(U10_mag.shape, -9999.0, dtype="float32")

valid_u = (U10_mag > -9999)

ustar[valid_u] = (k * U10_mag[valid_u]) / np.log(10 / z0)

print("ustar min/max:",
      float(ustar[ustar > -9999].min()),
      float(ustar[ustar > -9999].max()))

ustar min/max: 0.039815593510866165 0.7285817861557007


## 7. Compute aerodynamic resistance (rah, neutral)
$$
r_{ah} = \frac{ln(\frac{z_2}{z_1})}{ku_*}
$$
For SEBAL standard: $ 𝑧_2 = 2.0, 𝑧_1 = 0.1 m, and 𝑘 =0.41$

In [7]:
z2 = 2.0   # upper height (m)
z1 = 0.1   # lower height (m)

rah = np.full(ustar.shape, -9999.0, dtype="float32")

valid_rah = (ustar > 0)

rah[valid_rah] = np.log(z2 / z1) / (k * ustar[valid_rah])

print("rah min/max:",
      float(rah[rah > -9999].min()),
      float(rah[rah > -9999].max()))

# Save aerodynamic resistance raster (rah)

rah_out = os.path.join(out_dir, f"rah_firstpass_{date}_{hour_str}Z_{region}.tif")

write_raster(rah_out, profile, rah, ndata=-9999.0)

rah min/max: 10.02861213684082 183.5126190185547
Saved: 04_indices\rah_firstpass_20200119_16Z_MSDelta.tif


## 8. Compute calibrated dT raster
### 8.1 Extract masks


In [8]:
# example (edit names as needed)
hot_mask  = os.path.join(out_dir, f"hot_pixels_{date}_{hour_str}Z_{region}.tif")
cold_mask  = os.path.join(out_dir, f"cold_pixels_{date}_{hour_str}Z_{region}.tif")

rasters = {
    "LST":  f"{out_dir}/LST_{date}_{region}.tif",
    "Rn": f"{out_dir}/Rn_{date}_{hour_str}Z_{region}.tif",
    "G": f"{out_dir}/G_{date}_{hour_str}Z_{region}.tif",
    "NDVI": f"{out_dir}/NDVI_{date}_{region}.tif",
    "Albedo": f"{out_dir}/ALBEDO_{date}_{region}.tif"
}

# Dictionary to store anchor values
anchors = {}

for name, path in rasters.items():
    hot_val  = extract_mask_mean(path, hot_mask)    # mean over hot mask
    cold_val = extract_mask_mean(path, cold_mask)   # mean over cold mask

    # store values for later steps
    anchors[name] = {"hot": hot_val, "cold": cold_val}

    # keep print for verification
    print(name, "| HOT:", hot_val, "| COLD:", cold_val)

LST | HOT: 12.774807 | COLD: 5.237287
Rn | HOT: 337.96582 | COLD: 338.01315
G | HOT: 18.994051 | COLD: 6.5118184
NDVI | HOT: 0.05736474 | COLD: 0.61274743
Albedo | HOT: 0.08271693 | COLD: 0.09235789


### 8.2 Compute H at hot and cold pixels
We now compute sensible heat flux H at anchors:
$$
𝐻 = 𝑅_𝑛 − 𝐺 − 𝐿𝐸
$$

In SEBAL anchor logic:

Cold pixel, assume H ≈ 0

Hot pixel, assume LE ≈ 0, so:
$$
𝐻_{ℎot} = 𝑅_{𝑛,ℎ𝑜𝑡} − 𝐺_{ho𝑡}
$$

In [9]:
# 11.2 Convert LST to Kelvin for calibration
Ts_hot  = anchors["LST"]["hot"] + 273.15
Ts_cold = anchors["LST"]["cold"] + 273.15

# Net radiation at hot pixel (W/m²) from anchor extraction
Rn_hot = anchors["Rn"]["hot"] 

# Soil heat flux at hot pixel (W/m²)
G_hot  = anchors["G"]["hot"]

# Sensible heat flux at hot pixel (assume LE ≈ 0 for hot pixel)
H_hot = Rn_hot - G_hot  

# Sensible heat flux at cold pixel (assume H ≈ 0 for cold pixel)
H_cold = 0.0  

# Print results for verification before moving to dT calibration
print("Ts_hot:", Ts_hot)
print("Ts_cold:", Ts_cold)
print("H_hot:", H_hot)

Ts_hot: 285.92480697631834
Ts_cold: 278.3872870445251
H_hot: 318.97177


### 8.3 Solve linear dT relation

In SEBAL
$$
dT = a T_b + b
$$
where, 
$$
a = \frac{dT_{hot} - dT_{cold}}{T_{s,hot} - T_{s,cold}}
$$
$$
b = dT_{hot} - a\,T_{s,hot}
$$

For SEBAL anchor calibration:

- Cold pixel: $dT_{cold} = 0$
- Hot pixel: $dT_{hot}$ will be computed from the anchor energy balance

In [10]:
rah_hot = 110.0          # s/m (neutral initial assumption)
rho_cp = 1.25 * 1004     # J/m³/K

dT_hot = H_hot * rah_hot / rho_cp
dT_cold = 0.0

# correct order
a = (dT_hot - dT_cold) / (Ts_hot - Ts_cold)
b = dT_hot - a * Ts_hot

print("dT_hot:", dT_hot)
print("a:", a)
print("b:", b)

dT_hot: 27.95768512862612
a: 3.709135814115831
b: -1032.5762565713924


### 8.4 Compute calibrated dT raster
$$
dT_{raster} = aT_s + b
$$

In [11]:
# # Compute first-pass dT from the anchor-based linear relation
# and constrain to a nonnegative physical range before saving.
dT = np.full(LstC.shape, -9999.0, dtype="float32")

valid_dt = (LstC > -9999)
Ts_full = LstC + 273.15

dT[valid_dt] = a * Ts_full[valid_dt] + b

print("dT min/max:", float(dT[dT > -9999].min()), float(dT[dT > -9999].max()))

# Clamp first-pass dT to physical lower bound

dT2 = np.full(LstC.shape, -9999.0, dtype="float32")
dT2[valid_dt] = np.maximum(dT[valid_dt], 0.0)

print("Clamped dT min/max:",
      float(dT2[dT2 > -9999].min()),
      float(dT2[dT2 > -9999].max()))


dT min/max: -106.57623291015625 60.8861083984375
Clamped dT min/max: 0.0 60.8861083984375


### 8.5 Save calibrated dT raster

In [12]:
profile.update(dtype="float32", nodata=-9999.0)

dT_out = np.where(dT2 > -9999, dT2, -9999).astype("float32")

dT_path = f"{out_dir}/dT_{date}_{hour_str}Z_{region}.tif"

write_raster(raster_path=dT_path, profile=profile, value=dT_out)


Saved: 04_indices/dT_20200119_16Z_MSDelta.tif


## 9. Compute first-pass H raster
The sensible heat flux relation is:

$$
H = \rho \, C_p \, \frac{dT}{r_{ah}}
$$

So the aerodynamic resistance is:

$$
r_{ah} = \frac{\rho \, C_p \, dT}{H}
$$

In [13]:
H1 = np.full(dT2.shape, -9999.0, dtype="float32")

valid_H = (dT2 > -9999)
H1[valid_H] = rho_cp * dT2[valid_H] / rah_hot

print("H1 min/max:",
      float(H1[H1 > -9999].min()),
      float(H1[H1 > -9999].max()))

# Save H
H_out = os.path.join(out_dir, f"H_firstpass_{date}_{hour_str}Z_{region}.tif")

write_raster(H_out, profile, H1)

H1 min/max: 0.0 694.6550903320312
Saved: 04_indices\H_firstpass_20200119_16Z_MSDelta.tif


## 10. Monin–Obukhov Length L
$$
L = - \frac{\rho C_p T_a u_*^3}{k g H}
$$

In [14]:
g = 9.81

L = np.full(H1.shape, -9999.0, dtype="float32")

valid_L = (H1 != 0) & (ustar > 0) & (TA > -9999)

L[valid_L] = - (rho_cp * TA[valid_L] * (ustar[valid_L] ** 3)) / (k * g * H1[valid_L])

print("L min/max:",
      float(L[L > -9999].min()),
      float(L[L > -9999].max()))

L min/max: -9987.28125 3.333441972732544


## 11. Stability correction functions 
$$
x = (1 - 16 \frac {z_2}{L})^0.25
$$
$$
psi_m = ln(\frac{1+x}{2}) + ln(\frac{1+x^2}{2}) - 2 tan^{-1}x  + \frac{\pi}{2}
$$
$$
psi_h = 2ln(\frac{1+x^2}{2})
$$
### 11.1 Calculate correction functions

In [15]:
psi_m = np.zeros(L.shape, dtype="float32")
psi_h = np.zeros(L.shape, dtype="float32")

# heights
z2 = 2.0
z1 = 0.1

# --- unstable (L < 0)
unstable = (L < 0) & (L > -9999)

x = (1 - 16 * (z2 / L[unstable]))**0.25

psi_m[unstable] = (2*np.log((1+x)/2) + np.log((1+x**2)/2) - 2*np.arctan(x)  + np.pi/2)

psi_h[unstable] = 2*np.log((1+x**2)/2)

# --- stable (L > 0)
stable = (L > 0)

psi_m[stable] = -5 * (z2 / L[stable])
psi_h[stable] = -5 * (z2 / L[stable])

print("psi_m min/max:",
      float(psi_m[psi_m > -9999].min()),
      float(psi_m[psi_m > -9999].max()))
print("psi_h min/max:",
      float(psi_h[psi_h > -9999].min()),
      float(psi_h[psi_h > -9999].max()))

psi_m min/max: -3708.65234375 4.195610523223877
psi_h min/max: -3708.65234375 5.850485324859619


### 11.2 Stabilize L near zero before using psi
The Monin–Obukhov length  $𝐿$L appears in the stability correction functions $\psi_m$ and $\psi_h$ through terms like $𝑧/𝐿$.
When 𝐿 is very close to zero (near-neutral conditions), these terms become numerically unstable and can produce unrealistically large $\psi$ values.
To prevent this, small positive and negative values of L are constrained to ±0.1.This stabilizes the computation without significantly affecting the physical interpretation

*Unstable:*
$$
x = \left(1-\frac{16z}{L}\right)^{1/4}
$$
*Stable:*
$$
\psi =-5\frac{z}{L}
$$

In [16]:
L_safe = L.copy()

small_pos = (L_safe > 0) & (L_safe < 0.1)
small_neg = (L_safe < 0) & (L_safe > -0.1)

L_safe[small_pos] = 0.1
L_safe[small_neg] = -0.1

print("L_safe min/max:",
      float(L_safe[L_safe > -9999].min()),
      float(L_safe[L_safe > -9999].max()))
unstable = (L_safe < 0) & (L_safe > -9999)
x = (1 - 16 * (z2 / L_safe[unstable]))**0.25

psi_m[unstable] = (
    2*np.log((1+x)/2)
    + np.log((1+x**2)/2)
    - 2*np.arctan(x)
    + np.pi/2
)

psi_h[unstable] = 2*np.log((1+x**2)/2)

# --- stable (L > 0)
stable = (L_safe > 0)
psi_m[stable] = -5 * (z2 / L_safe[stable])
psi_h[stable] = -5 * (z2 / L_safe[stable])
print("psi_m range:", psi_m.min(), psi_m.max())
print("psi_h range:", psi_h.min(), psi_h.max())

L_safe min/max: -9987.28125 3.333441972732544
psi_m range: -100.0 3.0636773
psi_h range: -100.0 4.493772


### 11.3 Compute stability-corrected aerodynamic resistance

In [17]:
rah_corr = np.full(rah.shape, -9999.0, dtype="float32")

valid_corr = (ustar > 0)

rah_corr[valid_corr] = (np.log(z2 / z1) - psi_h[valid_corr]) / (k * ustar[valid_corr])

print("rah_corr min/max:",
      float(rah_corr[rah_corr > -9999].min()),
      float(rah_corr[rah_corr > -9999].max()))

rah_corr min/max: -91.7669448852539 6309.314453125


### 11.4 Fix nonphysical values before saving

In [18]:
rah_corr_fixed = rah_corr.copy()

# remove nonphysical values
rah_corr_fixed[rah_corr_fixed <= 0] = -9999

# cap extreme stable values
rah_corr_fixed[rah_corr_fixed > 2000] = 2000

print("rah_corr min/max:",
      float(rah_corr_fixed[rah_corr_fixed > -9999].min()),
      float(rah_corr_fixed[rah_corr_fixed > -9999].max()))

rahcorr_out = os.path.join(out_dir, f"rah_refined_{date}_{hour_str}Z_{region}.tif")

write_raster(rahcorr_out, profile, rah_corr_fixed, ndata=-9999.0)

rah_corr min/max: 0.022761717438697815 2000.0
Saved: 04_indices\rah_refined_20200119_16Z_MSDelta.tif


### 11.5 Recompute dT relation

In [19]:
# refined valid mask for anchor selection
anchor_valid = (
    (valid) &
    np.isfinite(Rn) & np.isfinite(G) &
    np.isfinite(LstC) & np.isfinite(Ndvi)
)

# use cleaned NDVI for anchor selection
ndvi_anchor = Ndvi2

# restrict to sparse vegetation / bare soil
hot_mask = anchor_valid & (ndvi_anchor <= 0.2)

print("Hot dry pixels:", int(hot_mask.sum()))

# pick hottest part of dry pixels
hot_LST_thr = np.nanpercentile(LstC[hot_mask], 90)
# Final hot-anchor pool:
# sparse vegetation/bare soil with highest LST,
# low NDVI, moderate G/Rn, consistent with dry hot surfaces.
hot_pixels = hot_mask & (LstC >= hot_LST_thr)
hot_pixels2 = hot_pixels & (Ndvi <= 0.15)

# corrected rah at hot anchor
rah_hot_corr = float(np.nanmean(rah_corr_fixed[hot_pixels]))

rho_cp = 1.25 * 1004   # J/m3/K

dT_hot_corr = H_hot * rah_hot_corr / rho_cp
dT_cold_corr = 0.0

a_corr = (dT_hot_corr - dT_cold_corr) / (Ts_hot - Ts_cold)
b_corr = dT_hot_corr - a_corr * Ts_hot

print("rah_hot_corr:", rah_hot_corr)
print("dT_hot_corr:", dT_hot_corr)
print("a_corr:", a_corr)
print("b_corr:", b_corr)

Hot dry pixels: 4060022
rah_hot_corr: 13.310582160949707
dT_hot_corr: 3.3830278630412707
a_corr: 0.4488250636355441
b_corr: -124.94719182308546


## 12. Corrected calibrated dT
### 12.1 Compute using coefficients

In [20]:
# Run after Notebook 5
dT_corr = np.full(LstC.shape, -9999.0, dtype="float32")

valid_dt = (LstC > -9999)

Ts_full = LstC + 273.15

dT_corr[valid_dt] = a_corr * Ts_full[valid_dt] + b_corr

print("dT_corr min/max:",
      float(dT_corr[dT_corr > -9999].min()),
      float(dT_corr[dT_corr > -9999].max()))

# clamp to physical lower bound
dT_corr2 = np.full(LstC.shape, -9999.0, dtype="float32")
dT_corr2[valid_dt] = np.maximum(dT_corr[valid_dt], 0.0)

print("Clamped dT_corr min/max:",
      float(dT_corr2[dT_corr2 > -9999].min()),
      float(dT_corr2[dT_corr2 > -9999].max()))

dT_corr min/max: -12.896286010742188 7.3675537109375
Clamped dT_corr min/max: 0.0 7.3675537109375


### 12.2 Save corrected dT raster

In [21]:
profile.update(dtype="float32", nodata=-9999.0)

dT_corr_out = np.where(dT_corr2 > -9999, dT_corr2, -9999).astype("float32")

dT_corr_path = f"{out_dir}/dT_corr_{date}_{hour_str}Z_{region}.tif"

write_raster(raster_path=dT_corr_path, profile=profile, value=dT_corr_out)

print("Saved:", dT_corr_path)
print("Min/Max saved:",
      float(dT_corr_out[dT_corr_out > -9999].min()),
      float(dT_corr_out[dT_corr_out > -9999].max()))

Saved: 04_indices/dT_corr_20200119_16Z_MSDelta.tif
Saved: 04_indices/dT_corr_20200119_16Z_MSDelta.tif
Min/Max saved: 0.0 7.3675537109375


## 13. Refined Sensible Heat H
The refined H using:
$$ 𝐻 = \rho 𝐶_𝑝 \frac{𝑑𝑇}{𝑟_{𝑎ℎ}}
$$
where: $\rho = .225 kg/m^3$ and $𝐶_𝑝 = 1004 J/kg/K$

### 13.1 Compute refined H

In [22]:
H_refined = np.full(dT_corr2.shape, -9999.0, dtype="float32")

valid_H2 = (dT_corr2 > -9999) & (rah_corr_fixed > 0)

H_refined[valid_H2] = rho_cp * dT_corr2[valid_H2] / rah_corr_fixed[valid_H2]

print("H_refined min/max:",
      float(H_refined[H_refined > -9999].min()),
      float(H_refined[H_refined > -9999].max()))

H_refined min/max: 0.0 51032.78515625


### 13.2 Recompute final H

In [23]:
rah_use = rah_corr_fixed.copy()
rah_use[(rah_use > 0) & (rah_use < 5.0)] = 5.0

H_refined = np.full(dT_corr2.shape, -9999.0, dtype="float32")

valid_H2 = (dT_corr2 > -9999) & (rah_use > 0)

H_refined[valid_H2] = rho_cp * dT_corr2[valid_H2] / rah_use[valid_H2]

print("rah_use min/max:",
      float(rah_use[rah_use > -9999].min()),
      float(rah_use[rah_use > -9999].max()))

print("H_refined min/max:",
      float(H_refined[H_refined > -9999].min()),
      float(H_refined[H_refined > -9999].max()))

rah_use min/max: 5.0 2000.0
H_refined min/max: 0.0 1067.9111328125


### 13.3 Save refined H

In [24]:
profile.update(dtype="float32", nodata=-9999.0)

H_refined_out = np.where(H_refined > -9999, H_refined, -9999).astype("float32")

Href_path = f"{out_dir}/H_refined_{date}_{hour_str}Z_{region}.tif"

write_raster(raster_path=Href_path, profile=profile, value=H_refined_out)

print("Saved:", Href_path)
print("Min/Max saved:",
      float(H_refined_out[H_refined_out > -9999].min()),
      float(H_refined_out[H_refined_out > -9999].max()))

Saved: 04_indices/H_refined_20200119_16Z_MSDelta.tif
Saved: 04_indices/H_refined_20200119_16Z_MSDelta.tif
Min/Max saved: 0.0 1067.9111328125


## 14. Compute first-pass LE raster
$$
LE = R_n − G − H_{refined}
$$
### 14.1 Compute LE using refined H

In [25]:
LE1 = np.full(H_refined.shape, -9999.0, dtype="float32")

valid_LE = (Rn > -9999) & (G > -9999) & (H_refined > -9999)

LE1[valid_LE] = Rn[valid_LE] - G[valid_LE] - H_refined[valid_LE]

print("LE1 min/max:",
      float(LE1[LE1 > -9999].min()),
      float(LE1[LE1 > -9999].max()))

LE1 min/max: -800.2109985351562 415.78106689453125


### 14.2 Clamp first-pass LE to a physical range

In [26]:
LE2 = np.full(LE1.shape, -9999.0, dtype="float32")
valid_LE2 = (LE1 > -9999)
LE2[valid_LE2] = np.maximum(LE1[valid_LE2], 0.0)

print("Clamped LE min/max:",
      float(LE2[LE2 > -9999].min()),
      float(LE2[LE2 > -9999].max()))

Clamped LE min/max: 0.0 415.78106689453125


### 14.3 Save clamped LE 

In [27]:
profile.update(dtype="float32", nodata=-9999.0)

LE_out = np.where(LE2 > -9999, LE2, -9999).astype("float32")

LE_path = f"{out_dir}/LE_firstpass_{date}_{hour_str}Z_{region}.tif"

write_raster(raster_path=LE_path, profile=profile, value=LE_out)

print("Saved:", LE_path)
print("Min/Max saved:",
      float(LE_out[LE_out > -9999].min()),
      float(LE_out[LE_out > -9999].max()))

Saved: 04_indices/LE_firstpass_20200119_16Z_MSDelta.tif
Saved: 04_indices/LE_firstpass_20200119_16Z_MSDelta.tif
Min/Max saved: 0.0 415.78106689453125


## 15. Evaporative fraction
$$
EF =  \frac{LE}{R_n - G}
$$
### 15.1 Compute first-pass evaporative fraction

In [28]:
EF1 = np.full(LE2.shape, -9999.0, dtype="float32")

denom = Rn - G
valid_EF = (LE2 > -9999) & (denom > 0)

EF1[valid_EF] = LE2[valid_EF] / denom[valid_EF]

print("EF1 min/max:",
      float(EF1[EF1 > -9999].min()),
      float(EF1[EF1 > -9999].max()))

EF1 min/max: 0.0 1.0


### 15.2 Save EF

In [29]:
profile.update(dtype="float32", nodata=-9999.0)

EF_out = np.where(EF1 > -9999, EF1, -9999).astype("float32")

EF_path = f"{out_dir}/EF_firstpass_{date}_{hour_str}Z_{region}.tif"

write_raster(raster_path=EF_path, profile=profile, value=EF_out)
print("Min/Max saved:",
      float(EF_out[EF_out > -9999].min()),
      float(EF_out[EF_out > -9999].max()))

Saved: 04_indices/EF_firstpass_20200119_16Z_MSDelta.tif
Min/Max saved: 0.0 1.0


## 16. Instantaneous ET equivalent (mm/hr)

------------------------------------------------------------
This section computes instantaneous evapotranspiration (ETinst) using the refined SEBAL energy balance results.

Steps followed:

1. Sensible heat flux (H) was refined using anchor-calibrated dT and stability-corrected aerodynamic resistance.
2. Latent heat flux (LE) was computed as: ($LE = R_n - G - H \$)
   
3. Evaporative fraction (EF) was derived as: $( EF = \frac{LE}{R_n - G}$)
4. Instantaneous ET is estimated as:
$$
ET_{inst} = EF \times ET_{eq}
$$
   where
$$
   ET_{eq} = \frac{(R_n - G)\times 3600}{\lambda}
$$
At this stage, ETinst is based on the refined energy balance solution and is expected to be physically more consistent than the initial first-pass estimate.
------------------------------------------------------------
### 16.1 Compute final instantaneous ET


In [30]:
lambda_v = 2.45e6

ETeq = np.full(Rn.shape, -9999.0, dtype="float32")
valid_eq = (Rn - G) > 0
ETeq[valid_eq] = (Rn[valid_eq] - G[valid_eq]) * 3600 / lambda_v

ETinst = np.full(EF1.shape, -9999.0, dtype="float32")
valid_inst = (EF1 > -9999) & (ETeq > -9999)

ETinst[valid_inst] = EF1[valid_inst] * ETeq[valid_inst]

ETinst[valid_inst] = np.clip(ETinst[valid_inst], 0.0, 1.5)

print("Final ETinst min/max:",
      float(ETinst[ETinst > -9999].min()),
      float(ETinst[ETinst > -9999].max()))

Final ETinst min/max: 0.0 0.610943615436554


### 16.2 Save final instantaneous ET raster

In [31]:
profile.update(dtype="float32", nodata=-9999.0)

ETinst_out = np.where(ETinst > -9999, ETinst, -9999).astype("float32")
ETinst_path = os.path.join(out_dir, f"ETinst_firstpass_{date}_{hour_str}_{region}.tif")

write_raster(raster_path=ETinst_path, profile=profile, value=ETinst_out)

Saved: 04_indices\ETinst_firstpass_20200119_16_MSDelta.tif


### 16.3 DAILY ET SCALING (SEBAL – FIRST PASS)
------------------------------------------------------------

Daily ET is estimated from final instantaneous ET by assuming evaporative fraction remains approximately stable during the daytime period. For this winter Mississippi Delta scene, a 9-hour effective daylight scaling is applied. Because this is still a first-pass SEBAL solution, daily ET is constrained to a practical physical range (0–8 mm/day) to limit residual numerical spikes before aerodynamic stability refinement.

------------------------------------------------------------

In [32]:
lambda_v = 2.45e6  # J/kg

# Rs_daily must be daily solar radiation (MJ/m²/day or W/m² integrated)
# If using daily Rn, replace below directly.

ET24 = np.full(ETinst.shape, -9999.0, dtype="float32")

valid_ET24 = (ETinst > -9999)

# Approximate daytime scaling factor for winter conditions
daylight_hours = 9.0

ET24[valid_ET24] = ETinst[valid_ET24] * daylight_hours

# Clamp to practical winter daily ET range
ET24[valid_ET24] = np.clip(ET24[valid_ET24], 0.0, 8.0)

print("Final ET24 min/max:",
      float(ET24[ET24 > -9999].min()),
      float(ET24[ET24 > -9999].max()))

Final ET24 min/max: 0.0 5.49849271774292


### 16.4 Save final daily ET raster

In [33]:
profile.update(dtype="float32", nodata=-9999.0)

ET24_out = np.where(ET24 > -9999, ET24, -9999).astype("float32")
ET24_path = f"{out_dir}/ET24_firstpass_{date}_{hour_str}Z_{region}.tif"

write_raster(raster_path=ET24_path, profile=profile, value=ET24_out)


Saved: 04_indices/ET24_firstpass_20200119_16Z_MSDelta.tif
